STAGE C1 — LOAD SILVER DATASETS

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [3]:
transactions = pd.read_csv(
    '../data/silver/transactions_clean.csv'
)

outlet_master = pd.read_csv(
    '../data/silver/outlet_master_clean.csv'
)

coords = pd.read_csv(
    '../data/silver/outlet_coordinates_clean.csv'
)

seasonality = pd.read_csv(
    '../data/silver/seasonality_clean.csv'
)

holidays = pd.read_csv(
    '../data/silver/holiday_clean.csv'
)

OUTLET-LEVEL BEHAVIORAL FEATURES

In [5]:
outlet_features = (
    transactions.groupby('Outlet_ID')
    .agg({
        'Volume_Liters': [
            'sum',
            'mean',
            'max',
            'std'
        ],

        'Total_Bill_Value': [
            'sum',
            'mean'
        ],

        'SKU_ID': 'nunique',

        'Distributor_ID': 'nunique'
    })
)

In [6]:
outlet_features.columns = [

    'Total_Volume',
    'Avg_Volume',
    'Max_Volume',
    'Volume_STD',

    'Total_Revenue',
    'Avg_Revenue',

    'SKU_Diversity',
    'Distributor_Diversity'
]

outlet_features = outlet_features.reset_index()

| Feature               | Meaning                    |
| --------------------- | -------------------------- |
| Total_Volume          | historical scale           |
| Avg_Volume            | typical purchase level     |
| Max_Volume            | possible uncapped behavior |
| Volume_STD            | demand volatility          |
| SKU_Diversity         | portfolio richness         |
| Distributor_Diversity | supply flexibility         |


CREATE MONTHLY AGGREGATION

In [8]:
monthly_outlet = (
    transactions.groupby([
        'Outlet_ID',
        'Year',
        'Month'
    ])['Volume_Liters']
    .sum()
    .reset_index()
)

In [9]:
active_months = (
    monthly_outlet.groupby('Outlet_ID')
    .size()
    .reset_index(name='Active_Months')
)

In [10]:
outlet_features = outlet_features.merge(
    active_months,
    on='Outlet_ID',
    how='left'
)

GROWTH TREND

In [11]:
monthly_outlet = monthly_outlet.sort_values([
    'Outlet_ID',
    'Year',
    'Month'
])

CALCULATE GROWTH

In [13]:
growth_features = []

for outlet_id, group in monthly_outlet.groupby('Outlet_ID'):

    group = group.sort_values(['Year', 'Month'])

    first = group['Volume_Liters'].iloc[0]
    last = group['Volume_Liters'].iloc[-1]

    growth = (
        (last - first) / first
        if first > 0 else 0
    )

    growth_features.append([
        outlet_id,
        growth
    ])

growth_df = pd.DataFrame(
    growth_features,
    columns=[
        'Outlet_ID',
        'Growth_Rate'
    ]
)

In [14]:
outlet_features = outlet_features.merge(
    growth_df,
    on='Outlet_ID',
    how='left'
)

SEASONALITY FEATURES

In [15]:
transactions['Date_Key'] = (
    transactions['Year'].astype(str)
    + '-'
    + transactions['Month'].astype(str)
)

In [17]:
monthly_sales = (
    transactions.groupby([
        'Outlet_ID',
        'Month'
    ])['Volume_Liters']
    .mean()
    .reset_index()
)

In [19]:
seasonality_strength = (
    monthly_sales.groupby('Outlet_ID')['Volume_Liters']
    .std()
    .reset_index(name='Seasonality_STD')
)

In [20]:
outlet_features = outlet_features.merge(
    seasonality_strength,
    on='Outlet_ID',
    how='left'
)

OPERATIONAL CAPACITY SIGNALS

In [21]:
outlet_features = outlet_features.merge(
    outlet_master,
    on='Outlet_ID',
    how='left'
)

In [22]:
outlet_features['Cooler_Count'] = (
    outlet_features['Cooler_Count']
    .fillna(0)
)

In [23]:
outlet_features['Has_Cooler'] = (
    outlet_features['Cooler_Count'] > 0
).astype(int)

SPATIAL INTELLIGENCE

In [ ]:
#!pip install osmnx geopandas

In [25]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point

CREATE GEO DATAFRAME

In [26]:
gdf_outlets = gpd.GeoDataFrame(
    coords,
    geometry=gpd.points_from_xy(
        coords.Longitude,
        coords.Latitude
    ),
    crs='EPSG:4326'
)

In [28]:
schools = ox.features_from_place(
    'Sri Lanka',
    tags={'amenity': 'school'}
)

/Users/akilafernando/Documents/GitHub/Data_Storm/venv/lib/python3.11/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 40 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [29]:
bus_stands = ox.features_from_place(
    'Sri Lanka',
    tags={'highway': 'bus_stop'}
)

/Users/akilafernando/Documents/GitHub/Data_Storm/venv/lib/python3.11/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 40 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [30]:
hospitals = ox.features_from_place(
    'Sri Lanka',
    tags={'amenity': 'hospital'}
)

/Users/akilafernando/Documents/GitHub/Data_Storm/venv/lib/python3.11/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 40 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [31]:
schools = schools.to_crs(epsg=3857)
bus_stands = bus_stands.to_crs(epsg=3857)
hospitals = hospitals.to_crs(epsg=3857)

gdf_outlets = gdf_outlets.to_crs(epsg=3857)

In [32]:
gdf_outlets['buffer'] = (
    gdf_outlets.geometry.buffer(500)
)

In [33]:
school_counts = []

for idx, row in gdf_outlets.iterrows():

    count = schools.within(
        row['buffer']
    ).sum()

    school_counts.append(count)

gdf_outlets['Nearby_Schools'] = school_counts

In [34]:
bus_counts = []

for idx, row in gdf_outlets.iterrows():

    count = bus_stands.within(
        row['buffer']
    ).sum()

    bus_counts.append(count)

gdf_outlets['Nearby_Bus_Stops'] = bus_counts

In [35]:
hospital_counts = []

for idx, row in gdf_outlets.iterrows():

    count = hospitals.within(
        row['buffer']
    ).sum()

    hospital_counts.append(count)

gdf_outlets['Nearby_Hospitals'] = hospital_counts

| POI        | Proxy Signal       |
| ---------- | ------------------ |
| Schools    | youth demand       |
| Bus stands | transient footfall |
| Hospitals  | continuous traffic |


In [36]:
spatial_features = gdf_outlets[[

    'Outlet_ID',
    'Nearby_Schools',
    'Nearby_Bus_Stops',
    'Nearby_Hospitals'

]]

outlet_features = outlet_features.merge(
    spatial_features,
    on='Outlet_ID',
    how='left'
)

In [37]:
cluster_features = outlet_features[[

    'Nearby_Schools',
    'Nearby_Bus_Stops',
    'Nearby_Hospitals',
    'Cooler_Count',
    'SKU_Diversity',
    'Avg_Volume'

]].fillna(0)

In [38]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    cluster_features
)

In [39]:
kmeans = KMeans(
    n_clusters=8,
    random_state=42
)

outlet_features['Cluster'] = (
    kmeans.fit_predict(X_scaled)
)

LATENT POTENTIAL ESTIMATION

FIND CLUSTER STARS

In [40]:
cluster_ceiling = (

    outlet_features
    .groupby('Cluster')['Avg_Volume']
    .quantile(0.95)
    .reset_index(name='Cluster_Potential_Ceiling')
)

In [41]:
outlet_features = outlet_features.merge(
    cluster_ceiling,
    on='Cluster',
    how='left'
)

POTENTIAL GAP

In [42]:
outlet_features['Potential_Gap'] = (

    outlet_features['Cluster_Potential_Ceiling']
    -
    outlet_features['Avg_Volume']
)

In [43]:
outlet_features['Estimated_Potential'] = (

    outlet_features['Avg_Volume']

    +

    (
        outlet_features['Potential_Gap']
        * 0.7
    )

)

Because:

not all gap is realistically capturable,
we avoid unrealistic overestimation.

This is:

conservative uncensoring.

In [44]:
outlet_features['Spatial_Score'] = (

    outlet_features['Nearby_Schools']

    +

    outlet_features['Nearby_Bus_Stops']

    +

    outlet_features['Nearby_Hospitals']
)

In [45]:
outlet_features['Estimated_Potential'] = (

    outlet_features['Estimated_Potential']

    *

    (
        1
        +
        (
            outlet_features['Spatial_Score']
            * 0.01
        )
    )

)

CREATING GOLD LAYER

In [46]:
gold_output = outlet_features[[

    'Outlet_ID',
    'Estimated_Potential'

]].copy()

In [47]:
gold_output.columns = [

    'Outlet_ID',
    'Maximum_Monthly_Liters'
]

In [49]:
gold_output.to_csv(
    '../data/gold/InferaX_predictions.csv',
    index=False
)

In [50]:
outlet_features.to_csv(
    '../data/gold/outlet_feature_table.csv',
    index=False
)